# Option Metrics Calculator

Loads an option chain CSV (produced by `fetch_opt_data.ipynb`) and computes:

- Intrinsic / extrinsic value, moneyness, break-even
- Full Black-Scholes Greeks (delta, gamma, vega, theta, rho) with dividend yield
- **P(ITM)** — risk-neutral probability of expiring in-the-money
- **P(profit)** — probability of making money *net of the premium paid*
- Greeks vs strike, volatility smile, and payoff diagram plots
- Portfolio-level Greek aggregation

All calculations live in `skew/utils.py`; this notebook only calls them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from skew.utils import bs_greeks_full, compute_option_metrics, portfolio_greeks
from skew import plot_strategy_distribution, OptionLeg

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Configuration

In [ ]:
CSV_PATH   = '../data/option_data.csv'   # update to your file
SYMBOL     = 'MSFT'
DEFAULT_RF = 0.04    # fallback risk-free rate when not in the CSV

## Load & prepare raw data

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
df_raw['symbol'] = SYMBOL

# yfinance stores option type as boolean (True = call); convert to string
if df_raw['option_type'].dtype == bool or df_raw['option_type'].isin([True, False]).all():
    df_raw['option_type'] = df_raw['option_type'].map({True: 'call', False: 'put'})

print(f"Rows loaded : {len(df_raw):,}")
print(f"Expiries    : {df_raw['expiry'].nunique()}")
df_raw.head()

## Compute all option metrics

`compute_option_metrics` handles validation, Greeks, and both probability columns in one call.

In [ ]:
result = compute_option_metrics(df_raw, default_rf=DEFAULT_RF)

display_cols = [
    'symbol', 'option_type', 'expiry', 'strike', 'underlying_price', 'option_price',
    'days_to_expiry', 'implied_vol',
    'intrinsic', 'extrinsic', 'moneyness', 'break_even', 'premium_pct', 'max_loss',
    'prob_itm', 'prob_profit',
    'delta', 'gamma', 'vega', 'theta_day', 'rho',
    'position_delta', 'position_gamma', 'position_vega', 'position_theta_day', 'position_rho',
]

result[display_cols].sort_values(['expiry', 'strike']).reset_index(drop=True).head(20)

## Portfolio-level Greeks

In [ ]:
port = portfolio_greeks(result)
pd.Series(port).to_frame('value')

## Probability of ITM vs Probability of Profit (net of premium)

- **P(ITM)** = `N(d2)` for calls, `N(-d2)` for puts — the risk-neutral probability the
  option expires in-the-money.
- **P(profit)** = probability the underlying moves *past the break-even*, i.e.
  `P(S_T > K + premium)` for calls and `P(S_T < K − premium)` for puts.

The shaded gap between the two curves is the **premium drag**: the probability of
being slightly ITM but not enough to recover the cost of the option.

In [ ]:
# pick the third-nearest expiry that still has live options
valid_expiries = (
    result[result['days_to_expiry'] > 0]['expiry']
    .sort_values().unique()
)
expiry_sample = valid_expiries[min(2, len(valid_expiries) - 1)]

sub   = result[result['expiry'] == expiry_sample]
calls = sub[sub['option_type'] == 'call'].sort_values('strike')
puts  = sub[sub['option_type'] == 'put'].sort_values('strike')
spot  = sub['underlying_price'].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df_leg, label in zip(axes, [calls, puts], ['Calls', 'Puts']):
    ax.plot(df_leg['strike'], df_leg['prob_itm'],
            lw=2, color='steelblue', label='P(ITM)')
    ax.plot(df_leg['strike'], df_leg['prob_profit'],
            lw=2, ls='--', color='darkorange', label='P(profit net of premium)')
    ax.fill_between(
        df_leg['strike'],
        df_leg['prob_profit'],
        df_leg['prob_itm'],
        alpha=0.15, color='gray', label='Premium drag'
    )
    ax.axvline(spot, color='gray', ls=':', lw=1.5, label=f'Spot {spot:.0f}')
    ax.set_title(f'{label}  —  {pd.Timestamp(expiry_sample).date()}')
    ax.set_xlabel('Strike')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=9)

fig.suptitle(
    f'{SYMBOL}  |  P(ITM) vs P(profit after premium paid)',
    fontsize=12
)
plt.tight_layout()
plt.show()

## Greeks profile across strikes

In [ ]:
c = calls[calls['days_to_expiry'] > 0].sort_values('strike')

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
fig.suptitle(
    f'{SYMBOL} Calls — Greeks vs Strike  ({pd.Timestamp(expiry_sample).date()})',
    fontsize=12
)

greek_cfg = [
    ('delta',     'Delta',             'tab:blue'),
    ('gamma',     'Gamma',             'tab:orange'),
    ('vega',      'Vega (per vol pt)', 'tab:green'),
    ('theta_day', 'Theta (per day)',   'tab:red'),
]

for ax, (col, ylabel, color) in zip(axes.flat, greek_cfg):
    ax.plot(c['strike'], c[col], lw=2, color=color)
    ax.axvline(spot, color='gray', ls=':', lw=1.2, label=f'Spot {spot:.0f}')
    ax.axhline(0, color='black', lw=0.7)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Strike')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Volatility smile by expiry

OTM puts (strike ≤ spot) and OTM calls (strike ≥ spot) stitched into a single smile per expiry.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

expiries_to_plot = result['expiry'].sort_values().unique()[:6]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(expiries_to_plot)))

for exp, color in zip(expiries_to_plot, colors):
    sub_e = result[result['expiry'] == exp]
    s0    = sub_e['underlying_price'].iloc[0]
    otm   = pd.concat([
        sub_e[(sub_e['option_type'] == 'put')  & (sub_e['strike'] <= s0)],
        sub_e[(sub_e['option_type'] == 'call') & (sub_e['strike'] >= s0)],
    ]).drop_duplicates('strike').sort_values('strike')
    otm = otm[otm['implied_vol'].between(0.01, 2.5)]
    if otm.empty:
        continue
    ax.plot(otm['strike'], otm['implied_vol'] * 100,
            lw=2, color=color, label=pd.Timestamp(exp).strftime('%Y-%m-%d'), alpha=0.9)

ax.axvline(result['underlying_price'].iloc[0], color='gray', ls=':',
           lw=1.5, label=f"Spot {result['underlying_price'].iloc[0]:.1f}")
ax.set_xlabel('Strike')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title(f'{SYMBOL} — Volatility Smile')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

## Payoff diagram — nearest ATM call & put

The risk-neutral terminal density is shown above; the expiry P&L below.
The green dotted vertical line marks the **break-even** (the point where
P&L turns positive). The area under the density curve to the right (call)
or left (put) of that line is **P(profit net of premium)**.

In [ ]:
# nearest ATM call with > 5 DTE
atm_pool = result[(result['option_type'] == 'call') & (result['days_to_expiry'] > 5)].copy()
atm_pool['dist'] = (atm_pool['strike'] - atm_pool['underlying_price']).abs()
row = atm_pool.sort_values('dist').iloc[0]

fig, axes, diag = plot_strategy_distribution(
    spot            = float(row['underlying_price']),
    time_to_expiry  = float(row['t_years']),
    risk_free_rate  = float(row['risk_free_rate']),
    dividend_yield  = float(row.get('div_yield', 0.0)),
    volatility      = float(row['implied_vol']),
    legs            = [OptionLeg('C', strike=float(row['strike']), premium=float(row['option_price']))],
    title           = f"{SYMBOL}  Long ATM Call  K={row['strike']:.0f}  {int(row['days_to_expiry'])} DTE",
)
print(f"P(ITM):                      {row['prob_itm']:.1%}")
print(f"P(profit net of premium):    {row['prob_profit']:.1%}")
print(f"Premium drag (ITM − profit): {row['prob_itm'] - row['prob_profit']:.1%}")
print(f"Expected payoff:             ${diag.expected_payoff:,.2f}")
if diag.breakeven_points:
    print(f"Break-even:                  ${diag.breakeven_points[0]:,.2f}")
plt.show()

In [ ]:
# nearest ATM put with > 5 DTE
atm_puts = result[(result['option_type'] == 'put') & (result['days_to_expiry'] > 5)].copy()
atm_puts['dist'] = (atm_puts['strike'] - atm_puts['underlying_price']).abs()
row_p = atm_puts.sort_values('dist').iloc[0]

fig, axes, diag_p = plot_strategy_distribution(
    spot            = float(row_p['underlying_price']),
    time_to_expiry  = float(row_p['t_years']),
    risk_free_rate  = float(row_p['risk_free_rate']),
    dividend_yield  = float(row_p.get('div_yield', 0.0)),
    volatility      = float(row_p['implied_vol']),
    legs            = [OptionLeg('P', strike=float(row_p['strike']), premium=float(row_p['option_price']))],
    title           = f"{SYMBOL}  Long ATM Put  K={row_p['strike']:.0f}  {int(row_p['days_to_expiry'])} DTE",
)
print(f"P(ITM):                      {row_p['prob_itm']:.1%}")
print(f"P(profit net of premium):    {row_p['prob_profit']:.1%}")
print(f"Premium drag (ITM − profit): {row_p['prob_itm'] - row_p['prob_profit']:.1%}")
print(f"Expected payoff:             ${diag_p.expected_payoff:,.2f}")
if diag_p.breakeven_points:
    print(f"Break-even:                  ${diag_p.breakeven_points[0]:,.2f}")
plt.show()

## Save results

In [ ]:
OUTPUT_PATH = '../data/option_metrics_output.csv'
(
    result[display_cols]
    .sort_values(['expiry', 'strike'])
    .reset_index(drop=True)
    .to_csv(OUTPUT_PATH, index=False)
)
print(f'Saved {len(result):,} rows → {OUTPUT_PATH}')